# U5.1 — Preparación de PySpark y exploración de NYC Taxi

**Materia:** Tópicos de Big Data  
**Objetivo:** comprobar PySpark en VS Code y explorar una muestra real de NYC Yellow Taxi sin guardar el dataset completo en la computadora.

**Alcance:** instalación, SparkSession, DataFrames y lectura remota de Parquet. Sin Databricks, limpieza ni modelos de ML.

**Fuente:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), Yellow Taxi, enero de 2025.

**Nota:** Spark se ejecuta localmente. La notebook consulta el Parquet por rangos HTTP y crea un archivo temporal con tres columnas del primer grupo de filas. El archivo temporal se elimina después de obtener la muestra.


## 1. Preparar y comprobar el entorno

En VS Code, selecciona el intérprete `U3_1_Vizualitation/.venv/Scripts/python.exe` como kernel. Si faltan paquetes, desde la carpeta `U3_1_Vizualitation` ejecuta en PowerShell:

```powershell
& .\.venv\Scripts\python.exe -m pip install pyspark ipykernel
```

Ejecuta la siguiente celda para comprobar el intérprete, PySpark y Java.

In [1]:
import os
from pathlib import Path
import subprocess
import sys

print("Intérprete:", sys.executable)

import pyspark

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)

java_home = os.environ.get("JAVA_HOME")
java_executable = str(Path(java_home) / "bin" / "java") if java_home else "java"
java = subprocess.run([java_executable, "-version"], capture_output=True, text=True, check=True)
print("Java:", (java.stderr or java.stdout).splitlines()[0])

Intérprete: c:\Users\mario\Downloads\Big_Data_GitHub\U3_1_Vizualitation\.venv\Scripts\python.exe
Python: 3.14.7
PySpark: 4.3.0
Java: openjdk version "21.0.12.1" 2026-08-18 LTS


## 2. Crear una SparkSession

`local[*]` utiliza los núcleos disponibles de esta computadora.


In [2]:
from pyspark.sql import SparkSession

# Los workers locales deben usar el mismo Python que el kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("U5_1_NYC_Taxi")
    .master("local[*]")
    .getOrCreate()
)

# La instrucción local[*] indica que Spark debe ejecutarse localmente, utilizando los núcleos disponibles de tu computadora.
# En Databricks no necesitaremos especificar master("local[*]"), porque la plataforma administra el entorno de ejecución.

print("Versión de Spark:", spark.version)
print("SparkSession creada correctamente")

Versión de Spark: 4.3.0
SparkSession creada correctamente


## 3. Probar un DataFrame pequeño

Comprobamos que Spark puede procesar registros y mostrar su esquema.


In [3]:
datos = [
    (1, 3.5, 15.0),
    (2, 8.2, 32.5),
    (3, 1.8, 10.0)
]

columnas = ["viaje_id", "distancia", "tarifa"]

df = spark.createDataFrame(datos, columnas)

df.show()
df.printSchema()

+--------+---------+------+
|viaje_id|distancia|tarifa|
+--------+---------+------+
|       1|      3.5|  15.0|
|       2|      8.2|  32.5|
|       3|      1.8|  10.0|
+--------+---------+------+

root
 |-- viaje_id: long (nullable = true)
 |-- distancia: double (nullable = true)
 |-- tarifa: double (nullable = true)



## 4. Conocer la fuente remota

Leemos el pie del archivo Parquet mediante solicitudes HTTP por rangos. La biblioteca Parquet incluida en Spark interpreta los metadatos y nos indica dónde están los datos de cada columna.

In [4]:
from urllib.request import Request, urlopen

URL_NYC = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

def leer_rango(inicio, fin):
    solicitud = Request(URL_NYC, headers={"Range": f"bytes={inicio}-{fin}"})
    with urlopen(solicitud, timeout=60) as respuesta:
        rango = respuesta.headers.get("Content-Range", "")
        if respuesta.status != 206 or rango.split("/")[0] != f"bytes {inicio}-{fin}":
            raise RuntimeError(f"Rango HTTP inesperado para {inicio}-{fin}: {rango}")
        datos = respuesta.read(fin - inicio + 1)
    if len(datos) != fin - inicio + 1:
        raise RuntimeError("La respuesta HTTP llegó incompleta.")
    return datos

with urlopen(Request(URL_NYC, method="HEAD"), timeout=30) as respuesta:
    tamano = int(respuesta.headers["Content-Length"])

pie = leer_rango(tamano - 8, tamano - 1)
longitud_metadatos = int.from_bytes(pie[:4], "little")
if pie[4:] != b"PAR1" or not 0 < longitud_metadatos <= tamano - 12:
    raise ValueError("El pie del Parquet no es válido.")

jvm = spark.sparkContext._jvm
parquet_util = jvm.org.apache.parquet.format.Util
pie_original = leer_rango(tamano - 8 - longitud_metadatos, tamano - 9)
metadatos = parquet_util.readFileMetaData(jvm.java.io.ByteArrayInputStream(pie_original))

print("Grupos de filas:", metadatos.getRow_groupsSize())
print("Filas totales declaradas:", metadatos.getNum_rows())
print("Columnas:", [campo.getName() for campo in list(metadatos.getSchema())[1:]])

Grupos de filas: 4
Filas totales declaradas: 3475226
Columnas: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


## 5. Leer una muestra sin descargar el dataset completo

Descargamos los bloques de tres columnas del primer grupo de filas. Con ellos construimos un Parquet temporal que Spark puede leer. La transferencia incluye los bloques completos de estas columnas. Spark conserva los primeros 1,000 registros en memoria antes de eliminar el archivo temporal.

In [5]:
from tempfile import TemporaryDirectory

columnas_nyc = ["VendorID", "trip_distance", "total_amount"]
muestra_metadata = metadatos.deepCopy()
grupo = muestra_metadata.getRow_groups().get(0)
bloques = jvm.java.util.ArrayList()
esquema = jvm.java.util.ArrayList()
campos = list(muestra_metadata.getSchema())
raiz = campos[0]
raiz.setNum_children(len(columnas_nyc))
esquema.add(raiz)
for campo in campos[1:]:
    if campo.getName() in columnas_nyc:
        esquema.add(campo)

with TemporaryDirectory(prefix="nyc_taxi_") as temporal:
    ruta = Path(temporal) / "muestra.parquet"
    with ruta.open("wb") as archivo:
        archivo.write(b"PAR1")
        for bloque in grupo.getColumns():
            columna = bloque.getMeta_data()
            if columna.getPath_in_schema().get(0) not in columnas_nyc:
                continue

            # Las páginas del diccionario, si existen, preceden a las páginas de datos.
            inicio_datos = columna.getData_page_offset()
            tiene_diccionario = columna.isSetDictionary_page_offset()
            inicio = columna.getDictionary_page_offset() if tiene_diccionario else inicio_datos
            longitud = columna.getTotal_compressed_size()
            nuevo_inicio = archivo.tell()
            archivo.write(leer_rango(inicio, inicio + longitud - 1))
            columna.setData_page_offset(nuevo_inicio + inicio_datos - inicio)
            if tiene_diccionario:
                columna.setDictionary_page_offset(nuevo_inicio)

            # El archivo temporal solo contiene páginas de datos y diccionario.
            bloque.setFile_offset(0)
            bloque.unsetColumn_index_offset()
            bloque.unsetColumn_index_length()
            bloque.unsetOffset_index_offset()
            bloque.unsetOffset_index_length()
            columna.unsetIndex_page_offset()
            columna.unsetBloom_filter_offset()
            columna.unsetBloom_filter_length()
            bloques.add(bloque)

        grupo.setColumns(bloques)
        grupo.unsetSorting_columns()
        grupo.setFile_offset(4)
        grupo.setTotal_compressed_size(archivo.tell() - 4)
        grupo.setTotal_byte_size(sum(b.getMeta_data().getTotal_uncompressed_size() for b in bloques))
        muestra_metadata.setSchema(esquema)
        muestra_metadata.unsetColumn_orders()
        muestra_metadata.unsetKey_value_metadata()
        muestra_metadata.setRow_groups(jvm.java.util.Collections.singletonList(grupo))
        muestra_metadata.setNum_rows(grupo.getNum_rows())
        salida = jvm.java.io.ByteArrayOutputStream()
        parquet_util.writeFileMetaData(muestra_metadata, salida)
        pie_muestra = bytes(salida.toByteArray())
        archivo.write(pie_muestra)
        archivo.write(len(pie_muestra).to_bytes(4, "little"))
        archivo.write(b"PAR1")

    # Materializar la muestra antes de eliminar el archivo temporal.
    registros = spark.read.parquet(ruta.as_uri()).select(*columnas_nyc).limit(1000)
    df_taxi = spark.createDataFrame(registros.collect(), registros.schema)

print("Registros de la muestra:", df_taxi.count())
df_taxi.printSchema()
df_taxi.show(5)

Registros de la muestra: 1000
root
 |-- VendorID: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)

+--------+-------------+------------+
|VendorID|trip_distance|total_amount|
+--------+-------------+------------+
|       1|          1.6|        18.0|
|       1|          0.5|       12.12|
|       1|          0.6|        12.1|
|       2|         0.52|         9.7|
|       2|         0.66|         8.3|
+--------+-------------+------------+
only showing top 5 rows


## 6. Primera exploración con PySpark

Utilizamos operaciones básicas: selección, filtro y agregación. **Los resultados corresponden a la muestra, no a todos los viajes de enero.**


In [ ]:
from pyspark.sql import functions as F

# Mostrar viajes con distancia positiva.
df_taxi.filter(F.col("trip_distance") > 0).show(5)

# Resumen sencillo de la muestra.
df_taxi.select(
    F.count("*").alias("viajes_muestra"),
    F.round(F.avg("trip_distance"), 2).alias("distancia_promedio"),
    F.round(F.avg("total_amount"), 2).alias("importe_promedio")
).show()


+--------+-------------+------------+
|VendorID|trip_distance|total_amount|
+--------+-------------+------------+
|       1|          1.6|        18.0|
|       1|          0.5|       12.12|
|       1|          0.6|        12.1|
|       2|         0.52|         9.7|
|       2|         0.66|         8.3|
+--------+-------------+------------+
only showing top 5 rows


## 7. Comprobación y reflexión

1. ¿Qué hace `SparkSession`?
2. ¿Cuál es la diferencia entre `df.show()` y `df.printSchema()`?
3. ¿Qué parte del archivo Parquet se consulta para encontrar los bloques de cada columna?
4. ¿Por qué no podemos afirmar que procesamos todo el dataset de NYC Taxi?
5. ¿En qué computadora se ejecutaron las operaciones de Spark?

**Resultado esperado:** una sesión de Spark funcional, metadatos del Parquet remoto y un DataFrame de muestra procesado localmente, sin descargar el archivo completo.
